# Jacobian — companion notebook

Companion for [`jacobian.md`](./jacobian.md). We visualize a nonlinear 2D map as a local linear transformation (its Jacobian) and verify the chain rule numerically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## 1. A nonlinear map and its Jacobian field

$\mathbf{f}(x,y) = (x^2 - y^2,\ 2xy)$ (the complex-squaring map). At each point, $J$ is a $2\times 2$ linear transformation; we draw it as the action on a small basis square.

In [ ]:
def f(p):
    x, y = p[0], p[1]
    return np.array([x**2 - y**2, 2*x*y])

def J(p):
    x, y = p[0], p[1]
    return np.array([[2*x, -2*y],[2*y, 2*x]])

# Grid of base points
xs = np.linspace(-1.5, 1.5, 7)
ys = np.linspace(-1.5, 1.5, 7)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ax1, ax2 = axes
for x in xs:
    for y in ys:
        p = np.array([x,y])
        ax1.plot(*p, 'k.', ms=3)
        fp = f(p)
        ax2.plot(*fp, 'k.', ms=3)
        # draw J e1, J e2 as arrows at p (shows local linearization)
        Jp = J(p) * 0.15
        ax1.arrow(p[0], p[1], Jp[0,0], Jp[1,0], color='tab:red', head_width=0.04, length_includes_head=True)
        ax1.arrow(p[0], p[1], Jp[0,1], Jp[1,1], color='tab:blue', head_width=0.04, length_includes_head=True)
ax1.set_aspect('equal'); ax1.set_title('Domain with J(p) columns shown'); ax1.set_xlim(-2, 2); ax1.set_ylim(-2, 2)
ax2.set_aspect('equal'); ax2.set_title('Image f(domain)'); ax2.set_xlim(-3, 3); ax2.set_ylim(-3, 3)
plt.show()

## 2. Chain rule check

If $h = g \circ f$ then $J_h(p) = J_g(f(p)) \cdot J_f(p)$. We check numerically against finite differences.

In [ ]:
def g(q):
    a, b = q[0], q[1]
    return np.array([np.sin(a)+b, a*b])

def Jg(q):
    a, b = q[0], q[1]
    return np.array([[np.cos(a), 1.0],[b, a]])

def numeric_jacobian(F, p, h=1e-6):
    n = len(p); m = len(F(p))
    Jn = np.zeros((m, n))
    for i in range(n):
        e = np.zeros(n); e[i] = h
        Jn[:, i] = (F(p + e) - F(p - e)) / (2*h)
    return Jn

p = np.array([0.7, 0.4])
h = lambda p: g(f(p))
Jh_chain = Jg(f(p)) @ J(p)
Jh_num   = numeric_jacobian(h, p)
print('chain-rule Jacobian:\n', Jh_chain)
print('\nfinite-diff Jacobian:\n', Jh_num)
print('\nrelative error:', np.linalg.norm(Jh_chain - Jh_num) / np.linalg.norm(Jh_chain))